# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json = dataset.metadata.to_json()
print(f"{metadata_json['name']}: {metadata_json['description']}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

Each record set, field, and column is referenced by its `@id`, as per the Croissant specification.

In [ ]:
# List available record sets and their field/column IDs
record_sets = list(dataset.metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    fields = rs.get('field', [])
    col_ids = []
    for field in fields:
        if isinstance(field, dict):
            field_id = field.get('@id', field)
        else:
            field_id = field
        # Find field metadata
        field_obj = next((f for f in dataset.metadata.fields if f['@id'] == field_id), None)
        if field_obj:
            columns = field_obj.get('column', [])
            col_ids.extend([c['@id'] if isinstance(c, dict) and '@id' in c else c for c in (columns if isinstance(columns, list) else [columns])])
            print(f"  Field: {field_id}")
            for c in (columns if isinstance(columns, list) else [columns]):
                col_id = c['@id'] if isinstance(c, dict) and '@id' in c else c
                print(f"    Column: {col_id}")
        else:
            print(f"  Field: {field_id}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** For this example, we'll extract data from the first available record set.

In [ ]:
# Extract data from each record set by their @id
dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

for record_set_id in record_set_ids:
    print(f"Processing record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for Record Set '{record_set_id}' with shape {df.shape}")
    else:
        print(f"No records found for Record Set '{record_set_id}'")

# Example: Show columns and head of the first DataFrame
if dataframes:
    first_rs_id = list(dataframes)[0]
    print(f"\nColumns in DataFrame for {first_rs_id}: {dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())
else:
    print('No DataFrames loaded. Check the record set availability in the metadata.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

Operations might include removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# Choose a record set and inspect its numeric fields
import numpy as np

if dataframes:
    rs_id = list(dataframes)[0]  # Example: The first record set
    df = dataframes[rs_id]
    # Try to infer a numeric field (e.g., 'MW [kDa]', 'MW', 'Coverage', etc.)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if not numeric_cols.empty:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field for EDA: {numeric_field}")
    else:
        # Try to coerce columns to numeric
        made_numeric = False
        for col in df.columns:
            coerced = pd.to_numeric(df[col], errors='coerce')
            if coerced.notna().sum() > 0:
                df[col+'_num'] = coerced
                numeric_field = col+'_num'
                print(f"Coerced column '{col}' to numeric for field '{numeric_field}'")
                made_numeric = True
                break
        if not made_numeric:
            print('No numeric fields found.')
            numeric_field = None
    
    if numeric_field:
        # Filter for values above median
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())
        # Normalization (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group by a categorical field if available
        potential_group_fields = df.select_dtypes(include=[object]).columns
        group_field = None
        for col in potential_group_fields:
            if df[col].nunique() < 20:
                group_field = col
                print(f"Grouping by field: {group_field}")
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field}:")
            display(grouped_df)
    else:
        print('No numeric field available for EDA')
else:
    print('No data available for EDA. Make sure dataframes are loaded.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

*Note: This example shows a histogram for the selected numeric field, and a boxplot grouped by a categorical field if available.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals() and numeric_field is not None:
    # Histogram
    plt.figure(figsize=(8,6))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    # If group_field is available
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"Boxplot of {numeric_field} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field or data available for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the `Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` dataset using the `mlcroissant` library. We inspected the metadata, listed record sets and fields (referenced by their `@id`), extracted records into pandas DataFrames, conducted basic exploratory data analysis, and visualized attribute distributions. This notebook provides a foundation for more advanced interrogation and modeling of mass spectrometry protein data.
